> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내** 🚨
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 03 & 04. 프롬프트 엔지니어링 기법 (새로운 시나리오 실습)</b></font>

## 0️⃣ 실습 전 프로젝트 Set Up
실습에 필요한 라이브러리를 설치합니다. 아래 블록을 실행해 설치를 완료하세요.

In [ ]:
!pip install -q uv
!uv pip install -q httpx==0.28.1 requests==2.32.4 langchain langsmith langchain_community langchain-openai anthropic langchain-anthropic langchain-google-genai google-ai-generativelanguage==0.6.15

In [ ]:
from google.colab import userdata
import os

OPENAI_API_KEY = (userdata.get('OPENAI_API_KEY')).strip()
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API Key 세팅 완료!")

In [ ]:
from openai import OpenAI
from enum import Enum

openai_client = OpenAI(api_key=OPENAI_API_KEY)

class OpenAIModel(Enum):
    GPT_35_TURBO = "gpt-3.5-turbo"
    GPT_4O_MINI = "gpt-4o-mini"

def openai_request(user_input: str,
                   system_prompt: str | None = None,
                   model: str = OpenAIModel.GPT_4O_MINI,
                   temperature: float = 0.5,
                   top_p: float = 1.0,
                   max_tokens: int = 1000):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    if user_input:
        messages.append({"role": "user", "content": user_input})

    completion = openai_client.chat.completions.create(
        model=model.value,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message

## 1️⃣ Zero-shot prompt engineering
---
### `실습1` 우주 탐사 기술 문서 번역
> 💫 **목표:** 기술 문서의 영한 번역 (단, 핵심 우주 기술 용어는 영어 원문을 유지하고 괄호 안에 한국어 뜻 병기)

In [ ]:
system_prompt = """
당신은 전문 기술 번역가입니다. 주어진 영문 텍스트를 한국어로 자연스럽게 번역하세요.
단, 기술적인 용어는 반드시 영어 원문을 그대로 유지하고 바로 뒤에 괄호 ( )를 사용하여 한국어 의미를 적어주세요. 
예시: Starship -> Starship(스타십)
"""
user_input = """
The new Starship architecture utilizes Orbital Refilling to achieve missions to Mars. 
These advanced capabilities provide new avenues for deep space exploration. 
In particular, performing Aerocapture maneuvers in the Martian atmosphere will significantly reduce fuel consumption.
"""

openai_result = openai_request(
    system_prompt=system_prompt,
    user_input=user_input,
    temperature=0.3
)
print(f"# OpenAI Result:\n{openai_result.content}")

## 2️⃣ Few-shot prompt engineering
---
### `실습2` 가상의 외계 과일/음식 사물 사전 정의하기
> 💫 **목표:** 주어진 가상의 단어 'Lumiberry'와 'Crondor'의 뜻과 예문을 퓨샷(Few-shot)을 활용해 생성하게 만들기

In [ ]:
system_prompt = """
A "Lumiberry" is a glowing blue fruit native to planet Xel.
An example of a sentence that uses the word Lumiberry is:
During the dark nights on Xel, we survived by picking and eating the sweet Lumiberries.

- "Crondor" means:
- An example of a sentence that uses the word Crondor is:
"""

openai_result = openai_request(
    user_input=system_prompt,
    temperature=0.7
)
print(f"# OpenAI Result:\n{openai_result.content}")

## 3️⃣ Chain-of-Thought Prompting (CoT)
---
### `실습3` 요리 기초 어휘를 가르쳐주는 AI 한국어 튜터
> 💫 **목표:** 프랑스어 화자 Sarah에게 한국어 요리 어휘(재료, 썰다, 끓이다)를 단계별로 가르쳐주는 멀티턴 AI 튜터 챗봇 만들기

In [ ]:
system_prompt = """
AI Tutor Role
=============
You are an AI Korean language tutor for Sarah, a French speaker who understands basic English.
Your primary goal is to teach her cooking vocabulary. Always wait for Sarah's input before proceeding.

Key Guidelines
--------------
1. Use beginner-level Korean. Provide French translations in parentheses for instructions.
2. Introduce one target word at a time.

Target Vocabulary:
- 재료 (ingrédients)
- 썰다 (couper)
- 끓이다 (bouillir)

Interaction Flow
----------------
1. Introduce the word and ask Sarah to repeat it.
2. Provide an example sentence in Korean.
3. Prompt Sarah to create her own sentence.
4. Move to the next word only after she understands.
"""

class LLMClient:
    def __init__(self, system_prompt=""):
        self.model = OpenAIModel.GPT_4O_MINI.value
        self.system_prompt = system_prompt
        self.msgs = []
    def send(self, user_prompt):
        if not self.msgs and self.system_prompt:
            self.msgs.append({"role":"system","content":self.system_prompt})
        self.msgs.append({"role":"user","content":user_prompt})
        r = openai_client.chat.completions.create(model=self.model, messages=self.msgs, temperature=0.5)
        text = r.choices[0].message.content.strip()
        self.msgs.append({"role":"assistant","content":text})
        return text

client = LLMClient(system_prompt=system_prompt)

print("💬 [실습용 간이 채팅] 'q'를 입력하면 종료됩니다.")
# 주석 해제 후 직접 테스트 해보세요!
# while True:
#     u = input("Sarah : ").strip()
#     if u.lower() in {"q","quit","exit"}: break
#     print("AI Tutor :", client.send(u), "\n")

## 4️⃣ Advanced Techniques: ReAct, ToT, Self-reflection, GoT
---
### `실습4` ReAct: 글로벌 시간 및 날씨 에이전트
> 💫 **목표:** 앞서 정의된 Tool을 사용하여 "파리의 현재 시간"과 "런던의 내일 날씨"를 찾아내는 에이전트 구동

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from geopy.geocoders import Nominatim
import requests, pytz, json, re
from datetime import datetime, timedelta
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate

geolocator = Nominatim(user_agent="weather-agent-new")

def clean_input(value: str) -> str:
    if not isinstance(value, str): return value
    value = value.strip().strip("'\"").strip()
    return re.sub(r'[\n\r\t]', '', value)

@tool("get_current_time", return_direct=False)
def get_current_time(timezone: str = "UTC") -> str:
    """지정된 타임존의 현재 시간을 반환합니다."""
    try:
        tz = pytz.timezone(clean_input(timezone))
        return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
    except Exception as e: return f"오류: {str(e)}"

@tool("get_tomorrow_date", return_direct=False)
def get_tomorrow_date(timezone: str = "UTC") -> str:
    """지정된 타임존의 내일 날짜를 반환합니다."""
    try:
        tz = pytz.timezone(clean_input(timezone))
        return (datetime.now(tz) + timedelta(days=1)).strftime("%Y-%m-%d")
    except Exception as e: return f"오류: {str(e)}"

class SearchInput(BaseModel):
    location: str = Field(description="도시명 (예: Paris, London)")
    date: str = Field(default=datetime.today().strftime("%Y-%m-%d"))

@tool("get_weather_forecast", args_schema=SearchInput, return_direct=False)
def get_weather_forecast(location: str, date: str = None) -> str:
    """Open-Meteo API를 사용해 날씨를 조회합니다."""
    location = clean_input(location)
    if location.startswith("{"):
        try:
            parsed = json.loads(location)
            location = clean_input(parsed.get("location", location))
            if "date" in parsed: date = clean_input(parsed["date"])
        except: pass
    date = datetime.today().strftime("%Y-%m-%d") if not date else clean_input(date)
    
    loc_obj = geolocator.geocode(location)
    if loc_obj:
        try:
            res = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={loc_obj.latitude}&longitude={loc_obj.longitude}&hourly=temperature_2m&start_date={date}&end_date={date}").json()
            sample = dict(list(zip(res["hourly"]["time"], res["hourly"]["temperature_2m"]))[:3])
            return f"{location} ({date}) 샘플 기온: {sample}"
        except Exception as e: return f"오류: {str(e)}"
    return "위치를 찾을 수 없습니다."

tools = [get_current_time, get_tomorrow_date, get_weather_forecast]

react_prompt = PromptTemplate.from_template("""
Answer questions using the tools provided:
{tools}
Tool names: {tool_names}
Format:
Question: input
Thought: thinking process
Action: tool name
Action Input: tool input
Observation: tool output
... (repeat N times)
Thought: I know the answer
Final Answer: answer in Korean
Question: {input}
Thought: {agent_scratchpad}""")

agent_executor = AgentExecutor(
    agent=create_react_agent(ChatOpenAI(model="gpt-4o-mini", temperature=0), tools, react_prompt),
    tools=tools, verbose=True, early_stopping_method="force", max_iterations=5
)

print("✅ ReAct 에이전트 테스트: 파리의 내일 날씨")
# result = agent_executor.invoke({"input": "프랑스 파리의 내일 날씨는 어때? 타임존은 Europe/Paris를 써줘."})

### `실습5` Self-Reflection: 리모트 워크 생산성 향상 포스트
> 💫 **목표:** 초안 작성 ➡️ 비평가(Critic)의 피드백 ➡️ 초안 수정의 흐름을 LangGraph로 구성하여 고품질의 SNS 게시글을 완성합니다.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from typing_extensions import TypedDict
from typing import Annotated

llm = ChatOpenAI(model="gpt-4o-mini")

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 HR 및 생산성 전문가입니다. 주어진 피드백을 반영하여 읽기 쉽고 매력적인 리모트 워크 생산성 향상 글을 작성하세요. 한국어로 작성하세요."),
    MessagesPlaceholder(variable_name="messages")
])
generator = gen_prompt | llm

critique_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 깐깐한 콘텐츠 에디터입니다. 주어진 글을 분석하고 더 나은 가독성, 매력적인 도입부, 행동 촉구(CTA)를 위한 실질적인 개선점 3가지를 비평하세요."),
    MessagesPlaceholder(variable_name="messages")
])
critic = critique_prompt | llm

class ContentState(TypedDict):
    messages: Annotated[list, add_messages]

async def create_node(state: ContentState):
    return {"messages": [await generator.ainvoke(state["messages"])]}

async def critique_node(state: ContentState):
    role_map = {"ai": HumanMessage, "human": AIMessage}
    transformed = [state["messages"][0]] + [role_map[m.type](content=m.content) for m in state["messages"][1:]]
    feedback = await critic.ainvoke(transformed)
    return {"messages": [HumanMessage(content=feedback.content)]}

def should_continue(state: ContentState):
    return END if len(state["messages"]) > 3 else "critique"

builder = StateGraph(ContentState)
builder.add_node("create_post", create_node)
builder.add_node("critique", critique_node)
builder.add_edge(START, "create_post")
builder.add_conditional_edges("create_post", should_continue, {"critique": "critique", END: END})
builder.add_edge("critique", "create_post")
workflow = builder.compile(checkpointer=InMemorySaver())

print("✅ Self-Reflection 워크플로우 구성 완료!")

### `실습6` Graph of Thought (GoT): IT 트러블슈팅 분석 에이전트
> 💫 **목표:** 증상 수집 ➡️ 개별 에러 분석 ➡️ 장애 원인 가설 ➡️ 종합 판단 ➡️ 해결책 정제의 복잡한 IT 지원 과정을 GoT 파이프라인으로 구성합니다.

In [ ]:
class TechSupportPrompter:
    @staticmethod
    def extract_issues_prompt() -> ChatPromptTemplate:
        sys = "IT 헬프데스크 기록자입니다. 고객의 불만사항에서 발생한 PC/네트워크 증상들을 추출하여 JSON 리스트로 만드세요."
        usr = "대화: {conversation}\n출력형식: {{\"issues\": [\"에러 증상 1\", \"에러 증상 2\"]}}"
        return ChatPromptTemplate.from_messages([("system", sys), ("user", usr)])

    @staticmethod
    def analyze_issue_prompt() -> ChatPromptTemplate:
        sys = "시스템 엔지니어입니다. 주어진 단일 IT 증상의 기술적 원인과 위험도를 분석하여 JSON으로 출력하세요."
        usr = "증상: {issue}\n출력형식: {{\"severity\": \"High|Medium|Low\", \"possible_causes\": [\"원인1\"], \"impact\": \"영향 요약\"}}"
        return ChatPromptTemplate.from_messages([("system", sys), ("user", usr)])

    @staticmethod
    def hypothesis_prompt() -> ChatPromptTemplate:
        sys = "IT 인프라 분석가입니다. 개별 증상 분석을 종합하여 가장 유력한 시스템 장애 원인 3가지를 JSON으로 도출하세요."
        usr = "증상들: {issues}\n분석결과: {analyses}\n출력형식: {{\"hypotheses\": [{{\"root_cause\": \"장애원인\", \"probability\": 90, \"reasoning\": \"이유\"}}]}}"
        return ChatPromptTemplate.from_messages([("system", sys), ("user", usr)])

    @staticmethod
    def aggregate_prompt() -> ChatPromptTemplate:
        sys = "수석 아키텍트입니다. 여러 가설을 종합해 최종적인 문제 진단과 트러블슈팅(해결) 우선순위 가이드를 작성하세요."
        usr = "가설 목록: {hypotheses}\n형식: 1. 최종 진단 2. 해결 단계 3. 요약"
        return ChatPromptTemplate.from_messages([("system", sys), ("user", usr)])

print("✅ IT Tech Support GoT 프롬프터 클래스 작성 완료! (기존 Medical 파이프라인의 구조를 그대로 재활용하여 사용할 수 있습니다.)")